In [24]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
# 다양성 
# 관련성
# 균형

In [2]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate, ChatPromptTemplate, FewShotChatMessagePromptTemplate

In [3]:
examples = [
    {"input" : "이 제품 정말 최고입니다!", "output" : "긍정"},
    {"input" : "품질이 너무 안좋아요", "output" : "부정"},
    {"input" : "보통이에요, 무난합니다.", "output" : "중립"},
    {"input" : "디자인은 좋은데 성능이 아쉬워요", "output" : "혼합"}
]

In [4]:
example_template = PromptTemplate(
    input_variables = ["input", "output"],
    template = "리뷰: {input}\n감정: {output}"
)

fewshot_prompt = FewShotPromptTemplate(
    examples = examples,
    example_prompt = example_template,
    prefix = '다음 예시를 참고해서 리뷰의 감정을 분류해주세요.\n',
    suffix = '\n리뷰: {input}\n감정:',
    input_variables = ['input'],
    example_separator = '\n\n'
)

In [5]:
formatted = fewshot_prompt.format(input = '가격은 비싸지만 품질이 뛰어나요')
print(formatted)

다음 예시를 참고해서 리뷰의 감정을 분류해주세요.


리뷰: 이 제품 정말 최고입니다!
감정: 긍정

리뷰: 품질이 너무 안좋아요
감정: 부정

리뷰: 보통이에요, 무난합니다.
감정: 중립

리뷰: 디자인은 좋은데 성능이 아쉬워요
감정: 혼합


리뷰: 가격은 비싸지만 품질이 뛰어나요
감정:


In [6]:
example_prompt_chat = ChatPromptTemplate.from_messages([
    ('human', '리뷰 : {input}'),
    ('ai', '감정: {output}')
])

fewshot_chat_prompt = FewShotChatMessagePromptTemplate(
    examples = examples,
    example_prompt = example_prompt_chat
)

In [7]:
final_prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 감정 분석 전문가입니다, 리뷰의 감정을 긍정/부정/중립/혼합 중 하나로 분류해주세요'),
    fewshot_chat_prompt,
    ('human', '리뷰 : {input}')
])

In [8]:
chain = final_prompt | llm 
result = chain.invoke({'input' : '가격은 비싸지만 품질이 뛰어나요'})
print(result.content)

감정: 혼합


In [9]:
large_examples = [
    {"input": "정말 만족합니다. 강력 추천!", "output": "긍정"},
    {"input": "배송이 빠르고 포장이 꼼꼼해요.", "output": "긍정"},
    {"input": "가격 대비 훌륭합니다.", "output": "긍정"},
    {"input": "품질이 최악입니다. 환불 요청했어요.", "output": "부정"},
    {"input": "고객센터 응대가 너무 불친절합니다.", "output": "부정"},
    {"input": "택배가 분실되었어요.", "output": "부정"},
    {"input": "보통이에요. 특별한 점은 없습니다.", "output": "중립"},
    {"input": "사진과 동일한 제품입니다.", "output": "중립"},
    {"input": "디자인은 예쁜데 내구성이 약해요.", "output": "혼합"},
    {"input": "기능은 많은데 사용법이 복잡합니다.", "output": "혼합"},
]


In [10]:
from langchain_core.example_selectors import SemanticSimilarityExampleSelector

In [11]:
from langchain_core.vectorstores import InMemoryVectorStore

In [15]:
example_selector = SemanticSimilarityExampleSelector.from_examples(
    large_examples,
    OpenAIEmbeddings(model="text-embedding-3-small"),
    InMemoryVectorStore,
    k=3
)

In [16]:
queries = [
    '배송이 너무 느려요',
    '값은 좀 나가지만 성능은 훌륭해요',
    '무난한 제품이에요'
]

for q in queries:
    selected = example_selector.select_examples({'input' : q})
    print(f"query : {q}")
    for s in selected:
        print(f" selected : {s['output']}, {s['input']}")

query : 배송이 너무 느려요
 selected : 긍정, 배송이 빠르고 포장이 꼼꼼해요.
 selected : 부정, 고객센터 응대가 너무 불친절합니다.
 selected : 부정, 택배가 분실되었어요.
query : 값은 좀 나가지만 성능은 훌륭해요
 selected : 긍정, 가격 대비 훌륭합니다.
 selected : 혼합, 기능은 많은데 사용법이 복잡합니다.
 selected : 혼합, 디자인은 예쁜데 내구성이 약해요.
query : 무난한 제품이에요
 selected : 중립, 사진과 동일한 제품입니다.
 selected : 부정, 품질이 최악입니다. 환불 요청했어요.
 selected : 긍정, 가격 대비 훌륭합니다.


In [ ]:
# 동적으로 선택 -> FewShotChatMessagePromptTemplate -> llm 답변

In [17]:
dynamic_few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_selector = example_selector,
    example_prompt = example_prompt_chat
)

dynamic_final_prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 감정 분석 전문가입니다. 리뷰의 감정을 긍정/부정/중립/혼합 중 하나로 분류하세요'),
    dynamic_few_shot_prompt,
    ('human', '리뷰 : {input}')
])

dynamic_chain = dynamic_final_prompt | llm


In [18]:
test_inputs  = ['택배 상자가 찌그러져왔어요', '화면은 선명한데 스피커 소리가 작아요']
for t in test_inputs:
    result = dynamic_chain.invoke({'input' : t})
    print(f'{t} : {result.content}')

택배 상자가 찌그러져왔어요 : 감정: 부정
화면은 선명한데 스피커 소리가 작아요 : 감정: 혼합


In [19]:
eval_dataset = [
    {"text": "완벽한 제품입니다! 강력 추천합니다.", "expected": "긍정"},
    {"text": "두 번 다시 구매하지 않겠습니다.", "expected": "부정"},
    {"text": "평범합니다. 특별히 좋지도 나쁘지도 않아요.", "expected": "중립"},
    {"text": "기능은 좋은데 AS가 아쉬워요.", "expected": "혼합"},
    {"text": "가격도 착하고 품질도 좋아요.", "expected": "긍정"},
    {"text": "사진과 너무 달라서 실망했습니다.", "expected": "부정"},
    {"text": "그냥 쓸만 합니다.", "expected": "중립"},
    {"text": "배송은 빠른데 제품이 기대 이하에요.", "expected": "혼합"},
]


In [20]:
def evaluate_zero_shot(dataset):
    results = []
    for item in dataset:
        response = llm.invoke([
            SystemMessage(content = "리뷰의 감정을 분류하세요. 반드시 '긍정', '부정', '중립', '혼합' 중 하나만 출력하세요"),
            HumanMessage(content = item['text'])
        ]).content
        
        predicted = response.strip()
        results.append({
            'text' : item['text'],
            'expected' : item['expected'],
            'predicted' : predicted,
            'correct' : predicted == item['expected'],
        })
        
    return results

In [25]:
def evaluate_few_shot(dataset, examples):
    results = []
    base_messages = [SystemMessage(content = "리뷰의 감정을 분류하세요. 반드시 '긍정', '부정', '중립', '혼합' 중 하나만 출력하세요")]
    for ex in examples:
        base_messages.append(HumanMessage(content = ex['input']))
        base_messages.append(AIMessage(content = ex['output']))
        
    for item in dataset:
        messages = base_messages + [HumanMessage(content = item['text'])]
        response = llm.invoke(messages).content
                
        predicted = response.strip()
        results.append({
            'text' : item['text'],
            'expected' : item['expected'],
            'predicted' : predicted,
            'correct' : predicted == item['expected'],
        })
        
    return results

In [26]:
few_shot_examples = [
    {"input": "정말 만족합니다. 강력 추천!", "output": "긍정"},
    {"input": "배송이 빠르고 포장이 꼼꼼해요.", "output": "긍정"},
    {"input": "가격 대비 훌륭합니다.", "output": "긍정"},
    {"input": "품질이 최악입니다. 환불 요청했어요.", "output": "부정"},
]

zero_results = evaluate_zero_shot(eval_dataset)

In [27]:
fewshot_results = evaluate_few_shot(eval_dataset, few_shot_examples)

In [28]:
zero_results

[{'text': '완벽한 제품입니다! 강력 추천합니다.',
  'expected': '긍정',
  'predicted': '긍정',
  'correct': True},
 {'text': '두 번 다시 구매하지 않겠습니다.',
  'expected': '부정',
  'predicted': '부정',
  'correct': True},
 {'text': '평범합니다. 특별히 좋지도 나쁘지도 않아요.',
  'expected': '중립',
  'predicted': '중립',
  'correct': True},
 {'text': '기능은 좋은데 AS가 아쉬워요.',
  'expected': '혼합',
  'predicted': '혼합',
  'correct': True},
 {'text': '가격도 착하고 품질도 좋아요.',
  'expected': '긍정',
  'predicted': '긍정',
  'correct': True},
 {'text': '사진과 너무 달라서 실망했습니다.',
  'expected': '부정',
  'predicted': '부정',
  'correct': True},
 {'text': '그냥 쓸만 합니다.', 'expected': '중립', 'predicted': '중립', 'correct': True},
 {'text': '배송은 빠른데 제품이 기대 이하에요.',
  'expected': '혼합',
  'predicted': '혼합',
  'correct': True}]

In [29]:
fewshot_results

[{'text': '완벽한 제품입니다! 강력 추천합니다.',
  'expected': '긍정',
  'predicted': '긍정',
  'correct': True},
 {'text': '두 번 다시 구매하지 않겠습니다.',
  'expected': '부정',
  'predicted': '부정',
  'correct': True},
 {'text': '평범합니다. 특별히 좋지도 나쁘지도 않아요.',
  'expected': '중립',
  'predicted': '중립',
  'correct': True},
 {'text': '기능은 좋은데 AS가 아쉬워요.',
  'expected': '혼합',
  'predicted': '혼합',
  'correct': True},
 {'text': '가격도 착하고 품질도 좋아요.',
  'expected': '긍정',
  'predicted': '긍정',
  'correct': True},
 {'text': '사진과 너무 달라서 실망했습니다.',
  'expected': '부정',
  'predicted': '부정',
  'correct': True},
 {'text': '그냥 쓸만 합니다.', 'expected': '중립', 'predicted': '중립', 'correct': True},
 {'text': '배송은 빠른데 제품이 기대 이하에요.',
  'expected': '혼합',
  'predicted': '혼합',
  'correct': True}]

In [30]:
import pandas as pd

In [35]:
pd.DataFrame(fewshot_results) # .to_csv('a.csv')

,text,expected,predicted,correct
0,완벽한 제품입니다! 강력 추천합니다.,긍정,긍정,True
1,두 번 다시 구매하지 않겠습니다.,부정,부정,True
2,평범합니다. 특별히 좋지도 나쁘지도 않아요.,중립,중립,True
3,기능은 좋은데 AS가 아쉬워요.,혼합,혼합,True
4,가격도 착하고 품질도 좋아요.,긍정,긍정,True
5,사진과 너무 달라서 실망했습니다.,부정,부정,True
6,그냥 쓸만 합니다.,중립,중립,True
7,배송은 빠른데 제품이 기대 이하에요.,혼합,혼합,True


In [ ]:
# classification related metric

In [ ]:
# 1.0      0.9    : root((1-0.9)**2) 

# 0.8,     0.5    :
# 0.9      0.1
# 0.0      0.2

In [ ]:
# regression related metric : rmse (root mean squared error) , mae mse...

In [ ]:
# 번역, 요약, 문구 생성 : BLUE, LOUGE
# 나는 학교에 갑니다 => I go to school  : I go school  0.75

In [ ]:
말이 25마리, 한번에 5마리만 경주를 할 수 있다
가장 빠른 3마리 말을 찾는게 목표
몇번의 경주를 해야될까요?

In [36]:
result = llm.invoke('말이 25마리 있습니다. 한 번에 5마리만 경주를 할 수 있습니다. 가장 빠른 3마리 말을 찾으려면 몇 번의 경주를 해야할까요?')

In [38]:
print(result.content)

25마리 말 중에서 가장 빠른 3마리를 찾기 위해서는 여러 번의 경주가 필요합니다. 이 문제를 해결하는 방법은 다음과 같습니다:

1. **첫 번째 경주**: 5마리씩 5개 그룹으로 나누어 총 5번 경주를 합니다. 각 경주에서 1위, 2위, 3위만 기록합니다.
   - 결과: 첫 번째 경주 후 5개의 그룹에서 각 그룹의 1위, 2위, 3위를 알고 있습니다.

2. **첫 번째 경주의 결과 정리**: 각 그룹의 1위 말, 2위 말, 3위 말 중 가장 빠른 말들을 뽑아내기 위해 다음 경주를 진행합니다.

3. **두 번째 경주**: 첫 번째 경주에서 1위를 차지한 5마리로 또 한 번 경주를 합니다.
   - 결과: 이 경주에서 1위의 말이 전체 1위라고 할 수 있습니다.

4. **가장 빠른 말 결정**: 두 번째 경주의 결과에 따라 빠른 3마리를 결정하기 위해 우리는 전체 결과를 살펴봐야 합니다. 이제 가장 빠른 말인 첫 번째 경주에서 1위 한 말을 기준으로 두 번째 경주의 성적을 확인합니다. 2위와 3위에 해당하는 말은 추가적으로 확인이 필요합니다.

5. **경주를 통해 대조**: 1위 마리와 함께 두 번째 경주와 첫 번째 경주에서 가까운 말들을 비교하여 가장 빠른 3마리를 결정합니다.

이 과정을 통해 총 대략 **7~8번의 경주**가 필요할 것입니다. 하지만 최적의 방식에 따라 실제로 필요한 경주 횟수는 달라질 수 있습니다. 한편으로, 1위 마리와 두 번째, 세 번째 자리를 찾기 위해 아래의 규칙을 적용합니다.

**결론**: 최종적으로 나올 경우, **최소 7번의 경주**로 가장 빠른 3마리를 구분할 수 있습니다.


In [39]:
result = llm.invoke('말이 6마리가 있고, 가장 빠른 말을 찾고 싶습니다. 한 번에 최대 6마리가 동시에 뛸 수 있습니다. 최소 몇 번의 경주가 필요할까요?')

In [40]:
print(result.content)

6마리의 말이 있을 때, 모든 말을 한 번에 다 경주할 수 있으므로, 가장 빠른 말을 찾는 최소 경주는 다음과 같이 진행할 수 있습니다:

1. **1차 경주**: 6마리를 한 번에 경주 시켜서 각 말의 위치를 비교합니다.
   - 이 경주로 각 말의 상대적인 순위를 알 수 있습니다. (예: 1위, 2위, 3위, 4위, 5위, 6위)

2. 경주 결과에 따라 가장 빠른 말이 1위로 나온 말을 바로 정할 수 있습니다.

따라서, 6마리의 말 중 가장 빠른 말을 찾기 위해 필요한 최소 경주는 **1번**입니다.


In [41]:
llm.invoke('3개 문 중 하나에 황금이 있고, 나머지는 야채입니다. 당신이 1번 문을 고르고, 사회자가 2번 문으로 바꾸시겠습니까? 라고 묻습니다.사회자는 아직 어떤 문도 열지 않았습니다. 바꾸는 것이 유리할까요?')

AIMessage(content='이 상황은 "몬티 홀 문제"라고 불리는 유명한 확률 문제와 유사합니다. 문제를 요약하자면, 3개의 문 중 하나에는 황금이 있고 나머지 2개 문에는 야채가 있습니다. 당신이 1번 문을 선택했을 때, 사회자는 다른 문(예: 2번 문)을 언급하며 이를 바꿀 것인지 물어봅니다.\n\n이 문제에서 중간에 문을 열지 않고 바꾸는 것이 유리한지 분석해보겠습니다.\n\n1. 당신이 처음에 1번 문을 선택했을 때, 그 문에 황금이 있을 확률은 1/3입니다. 나머지 두 문(2번, 3번)에 황금이 있을 확률의 합은 2/3입니다.\n\n2. 사회자가 2번 문으로 바꾸는 방법은 황금이 2번 문에 없을 때만 가능합니다. 따라서 만약 1번 문에 황금이 없다면, 사회자는 항상 3번 문을 열어줍니다. \n\n결론적으로,\n- 바꾸지 않으면 황금이 있을 확률은 1/3입니다.\n- 바꾸면 황금이 있을 확률은 2/3입니다.\n\n따라서, 문을 바꾸는 것이 훨씬 유리합니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 290, 'prompt_tokens': 74, 'total_tokens': 364, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ca3e7d71bf', 'id': 'chatcmpl-DNe2QBAHbPeWxMsTGcTyzUvm2t8F5', 'se

In [42]:
print('이 상황은 "몬티 홀 문제"라고 불리는 유명한 확률 문제와 유사합니다. 문제를 요약하자면, 3개의 문 중 하나에는 황금이 있고 나머지 2개 문에는 야채가 있습니다. 당신이 1번 문을 선택했을 때, 사회자는 다른 문(예: 2번 문)을 언급하며 이를 바꿀 것인지 물어봅니다.\n\n이 문제에서 중간에 문을 열지 않고 바꾸는 것이 유리한지 분석해보겠습니다.\n\n1. 당신이 처음에 1번 문을 선택했을 때, 그 문에 황금이 있을 확률은 1/3입니다. 나머지 두 문(2번, 3번)에 황금이 있을 확률의 합은 2/3입니다.\n\n2. 사회자가 2번 문으로 바꾸는 방법은 황금이 2번 문에 없을 때만 가능합니다. 따라서 만약 1번 문에 황금이 없다면, 사회자는 항상 3번 문을 열어줍니다. \n\n결론적으로,\n- 바꾸지 않으면 황금이 있을 확률은 1/3입니다.\n- 바꾸면 황금이 있을 확률은 2/3입니다.\n\n따라서, 문을 바꾸는 것이 훨씬 유리합니다.')

이 상황은 "몬티 홀 문제"라고 불리는 유명한 확률 문제와 유사합니다. 문제를 요약하자면, 3개의 문 중 하나에는 황금이 있고 나머지 2개 문에는 야채가 있습니다. 당신이 1번 문을 선택했을 때, 사회자는 다른 문(예: 2번 문)을 언급하며 이를 바꿀 것인지 물어봅니다.

이 문제에서 중간에 문을 열지 않고 바꾸는 것이 유리한지 분석해보겠습니다.

1. 당신이 처음에 1번 문을 선택했을 때, 그 문에 황금이 있을 확률은 1/3입니다. 나머지 두 문(2번, 3번)에 황금이 있을 확률의 합은 2/3입니다.

2. 사회자가 2번 문으로 바꾸는 방법은 황금이 2번 문에 없을 때만 가능합니다. 따라서 만약 1번 문에 황금이 없다면, 사회자는 항상 3번 문을 열어줍니다. 

결론적으로,
- 바꾸지 않으면 황금이 있을 확률은 1/3입니다.
- 바꾸면 황금이 있을 확률은 2/3입니다.

따라서, 문을 바꾸는 것이 훨씬 유리합니다.


In [ ]:
# Sally(여자) 한테 남자 형제가 3명 있어요
# 각 남자형제한테는 여자 형제가 2명 있어요
# Sally 의 여자 형제는 몇명?

In [43]:
print(llm.invoke('LOLLAPALOOZA 라는 단어에서 L이 몇 번 나타나나요?').content)

"LOLLAPALOOZA"라는 단어에서 L은 총 3번 나타납니다.


In [ ]:
# print(llm.invoke('LOLLAPALOOZA 라는 단어에서 L이 몇 번 나타나나요?').content)

In [44]:
# CoT (Chain-of-Thought)
# "단계별로 생각해보세요"

In [45]:
result = llm.invoke('말이 6마리가 있고, 가장 빠른 말을 찾고 싶습니다. 한 번에 최대 6마리가 동시에 뛸 수 있습니다. 최소 몇 번의 경주가 필요할까요?')

In [47]:
llm.invoke('3개 문 중 하나에 황금이 있고, 나머지는 야채입니다. 당신이 1번 문을 고르고, 사회자가 2번 문으로 바꾸시겠습니까? 라고 묻습니다.사회자는 아직 어떤 문도 열지 않았습니다. 바꾸는 것이 유리할까요? 단계별로 생각해봅시다')

AIMessage(content='이 문제는 "몬티 홀 문제"라고 불리는 확률 문제와 관련이 있습니다. 단계별로 살펴보겠습니다.\n\n1. **초기 상태**:\n   - 문 1: 당신이 선택한 문\n   - 문 2: 사회자가 열지 않는 문\n   - 문 3: 잠재적으로 금이 있을 수 있는 문\n\n   황금이 들어있는 문이 있을 확률:\n   - 문 1에 황금이 있을 확률: 1/3\n   - 문 2에 황금이 있을 확률: 1/3\n   - 문 3에 황금이 있을 확률: 1/3\n\n2. **사회자의 역할**:\n   - 사회자는 당신이 선택하지 않은 문 중에서 야채가 있는 문을 열어야 합니다. 즉, 문 2와 문 3 중에서 하나를 열게 되는데, 당신이 문 1을 선택했으므로, 문 2나 문 3 중 한 문은 반드시 야채가 들어있는 문이 됩니다.\n\n3. **구체적인 경우의 수**:\n   - 경우 1: 문 1에 황금이 있음 (확률 1/3)\n     - 사회자는 문 2 또는 문 3 중 야채가 있는 문을 열게 됩니다.\n   - 경우 2: 문 2에 황금이 있음 (확률 1/3)\n     - 사회자는 문 3를 열게 됩니다.\n   - 경우 3: 문 3에 황금이 있음 (확률 1/3)\n     - 사회자는 문 2를 열게 됩니다.\n\n4. **선택 변경 시 확률**:\n   - 만약 당신이 문 1을 선택하고, 사회자가 문 2를 열었을 때, 문 1에 황금이 있을 확률은 여전히 1/3입니다.\n   - 문 3에 황금이 있을 경우, 사회자가 문 2를 열었으므로 문 3에 황금이 있을 확률은 2/3입니다.\n\n5. **결론**:\n   - 따라서, 문 1에서 문 3으로 바꾸는 것이 유리합니다. 문 1에 머무를 경우 황금이 있을 확률은 1/3이고, 문 3으로 바꿀 경우 황금이 있을 확률은 2/3이므로, 바꾸는 것이 더 나은 선택입니다. \n\n결론적으로, 문 1을 버리고 문 3으로 바꾸는 것이 확률적으로 유리합니다.', additional_kwargs={'refusal': None}

In [48]:
print(llm.invoke('LOLLAPALOOZA 라는 단어에서 L이 몇 번 나타나나요? 단계별로 생각해보세요').content)

"LOLLAPALOOZA"라는 단어에서 'L'이 몇 번 나타나는지 단계별로 살펴보겠습니다.

1. **단어 나열하기**: L O L L A P A L O O Z A
2. **L 세기**: 각 글자를 하나씩 확인합니다.
   - L → 1회
   - O → (무시)
   - L → 2회
   - L → 3회
   - A → (무시)
   - P → (무시)
   - A → (무시)
   - L → 4회
   - O → (무시)
   - O → (무시)
   - Z → (무시)
   - A → (무시)

3. **결과 도출**: 'L'은 총 4번 나타납니다.

따라서 "LOLLAPALOOZA"에서 'L'은 4번 나타납니다.


In [50]:
print(llm.invoke('3개 문 중 하나에 황금이 있고, 나머지는 야채입니다. 당신이 1번 문을 고르고, 사회자가 2번 문으로 바꾸시겠습니까? 라고 묻습니다.사회자는 아직 어떤 문도 열지 않았습니다. 바꾸는 것이 유리할까요? 단계별로 생각해봅시다').content)

이 문제는 "몬티 홀 문제"로 알려진 것으로, 사람들에게 자주 오해를 불러일으키는 확률 문제입니다. 단계별로 생각해보겠습니다.

1. **초기 선택**: 시작할 때, 3개의 문(1번, 2번, 3번) 중에서 하나를 선택합니다. 당신은 1번 문을 선택했습니다.

2. **문 뒤의 배치**: 문 뒤에는 한 개의 황금과 두 개의 야채가 있습니다. 각각의 문에 황금이 있을 확률은 다음과 같습니다:
   - 1번 문: 황금이 있을 확률 1/3
   - 2번 문: 황금이 있을 확률 1/3
   - 3번 문: 황금이 있을 확률 1/3

3. **사회자의 행동**: 사회자는 항상 당신이 선택하지 않은 문 중에서 야채가 있는 문을 열게 됩니다. 당신이 1번 문을 선택했기 때문에, 그 다음 사회자는 2번 문이나 3번 문 중 하나를 열어야 합니다. 만약 2번 문에 야채가 있을 경우, 그는 3번 문을 열고 3번 문에 야채가 있을 경우 2번 문을 엽니다.

4. **선택한 문과 열릴 문 분석**:
   - 당신이 1번 문을 선택했을 때, 세 가지 경우가 있습니다:
     1. **황금이 1번 문에 있는 경우**: (확률 1/3)
        - 사회자는 2번 문 또는 3번 문 중에서 야채가 있는 문을 엽니다.
     2. **황금이 2번 문에 있는 경우**: (확률 1/3)
        - 사회자는 2번 문을 열 수 없으므로 3번 문을 엽니다.
     3. **황금이 3번 문에 있는 경우**: (확률 1/3)
        - 사회자는 3번 문을 열 수 없으므로 2번 문을 엽니다.

5. **선택 변경**:
   - 만약 당신이 선택을 변경하면:
     - 황금이 1번 문에 있는 경우(확률 1/3): 변경하면 패배
     - 황금이 2번 문에 있는 경우(확률 1/3): 변경하면 승리
     - 황금이 3번 문에 있는 경우(확률 1/3): 변경하면 승리
   - 따라서, 선택을 변경할 경우의 승리 확률은 2/3이고, 유지할 경우의 승리 확률은 1/3입니다.

결론적으로, 

In [51]:
print(llm.invoke("""Sally(여자) 한테 남자 형제가 3명 있어요
각 남자형제한테는 여자 형제가 2명 있어요
Sally 의 여자 형제는 몇명인가요?
단계별로 생각해보세요""").content)

먼저 주어진 정보를 정리해보겠습니다.

1. Sally는 여자입니다.
2. Sally에게는 남자 형제가 3명 있습니다. (이 남자 형제들은 각각 Sally의 형제입니다.)
3. 각 남자 형제마다 여자 형제가 2명 있습니다.

이제 단계별로 생각해보겠습니다.

- Sally는 한 명이고, Sally는 자신을 포함한 여자 형제를 셀 수 있습니다.
- Sally의 남자 형제 3명에게는 모두 2명의 여자 형제가 있습니다. 여기서 "여자 형제"가 Sally를 포함하는지 확인해야 합니다.

만약 각 남자 형제가 가리키는 여자 형제가 Sally를 포함하는 것이라면, 그들의 여자 형제는 다음과 같습니다:

- Sally (형제의 여자 형제 중 1명)
- 그리고 이 형제들마다 각기 1명의 다른 여자 형제가 있어야 할 것입니다.

그러나 Sally에게는 남자 형제가 3명밖에 없고, 각 남자 형제마다 Sally라는 여자형제가 있기 때문에, 추가적인 여자형제는 없습니다. 따라서 Sally의 남자 형제는 Sally 1명과 그 이외의 여자형제가 없습니다.

결론적으로, Sally의 여자 형제는 1명(자기 자신)입니다. 따라서 최종 결과는:

**Sally의 여자 형제는 1명입니다.**


In [ ]:
# CoT
# 단계별로 생각해보세요, 봅시다
# 지문의 조건을 전부 나열 - 각각의 조건들을 검토하면서 추론해줘/풀어줘
# 이 문제가 기존 유명한 문제와 어떻게 다른지를 분석하고 답해줘

In [52]:
print(llm.invoke('3개 문 중 하나에 황금이 있고, 나머지는 야채입니다. 당신이 1번 문을 고르고, 사회자가 2번 문으로 바꾸시겠습니까? 라고 묻습니다.사회자는 아직 어떤 문도 열지 않았습니다. 바꾸는 것이 유리할까요? 단계별로 생각해봅시다, 이 문제가 기존 유명한 문제와 어떻게 다른지를 분석하고 답해줘').content)

이 문제는 유명한 "몬티 홀 문제"와 유사한 구조를 가지고 있으나, 약간의 변형이 있습니다. 몬티 홀 문제에서 사회자는 항상 당신이 선택하지 않은 문 중에 하나를 열어 보이고, 그 문 뒤에는 항상 야채가 있습니다. 그러나 당신의 질문에서는 사회자가 처음에 어떤 문도 열지 않았기 때문에 이 부분이 다릅니다.

### 문제 분석

1. **상황 설정**:
   - 3개의 문 (1번, 2번, 3번) 중 하나에 황금이 있습니다.
   - 나머지 2개의 문은 야채입니다.

2. **첫 번째 선택**:
   - 당신이 1번 문을 선택했습니다.

3. **사회자의 행동**:
   - 사회자는 "2번 문으로 바꾸시겠습니까?"라고 묻습니다.
   - 현재 사회자는 아무 문도 열지 않았고, 2번 문을 제안했습니다.

### 확률 평가

1. **처음 선택 확률**:
   - 당신이 1번 문을 선택했을 때, 황금이 있을 확률은 1/3입니다.
   - 당신이 선택하지 않은 2번과 3번 중 하나에 황금이 있을 확률은 2/3입니다.

2. **사회자의 제안**:
   - 사회자가 무작위로 2번 문으로 바꾸라는 것이기 때문에, 이 제안은 다음과 같은 것을 의미합니다:
     - 만약 1번 문에 황금이 있다면, 2번 문은 야채이고, 사회자는 2번 문을 제안할 수 없습니다.
     - 만약 황금이 2번 문에 있다면, 당신은 1번 문에서 2번 문으로 바꾸는 것이 유리해집니다.
     - 만약 황금이 3번 문에 있다면, 2번 문을 제안할 수 없습니다.

### 결론

- 당신이 1번 문을 고른 후 사회자가 2번 문으로 바꾸면, 단순히 그 문이 유리한지를 따질 수 있습니다.
- 전체적으로, 사회자가 문을 열지 않았기 때문에 "바꾸기"가 반드시 유리하다고 말할 수는 없습니다. 선택한 문(1번)과 제안된 문(2번) 모두 황금이 있을 확률은 1/3입니다. 

결과적으로, 당신이 바꾸는 것이 유리할 가능성이 몬티 홀 문제의 확률적인 구조와는 다르기 때문에, 전반적으로 바꾼다고 해서 반드시 유리하다

In [ ]:
# few-shot cot : CoT, 예시

In [ ]:
system_prompt = """당신은 퍼즐 전문가입니다. 아래 문제들은 유명한 문제의 **변형**입니다.
원래 문제의 답을 그대로 적용하지 마세요. 이 문제의 조건을 정확히 읽고 답하세요.

예시:
문제 : 셔츠 1장을 말리는데 4시간이 걸립니다. 셔츠 5장을 말리면 몇 시간?
[흔한 실수] 4 x 5 = 20시간
[올바른 생각] 동시에 널면 됩니다. 답 : 4시간

이처럼 문제의 조건을 주의 깊게 읽고 단계별로 생각하세요
"""

In [ ]:
"친구 집까지 평균 시속 3마일로 걸어갔습니다. 왕복 전체 평균 속도를 시속 6마일로 만들려면, 돌아올 때 얼마나 빨리 달려야 할까요?"

In [53]:
prompt = """
당신은 물리 문제를 푸는 전문가입니다. 절대 직감이나 추측으로 답하지 말고, 정의에 따라 단계적으로 계산하세요.

문제 :
한 사람이 친구 집까지 평균 시속 3마일로 걸어갔습니다. 왕복 전체 평균 속도를 시속 6마일로 만들려면, 돌아올 때 얼마나 빨리 달려야 할까요?

다음 순서를 반드시 따르세요

1. 편도 거리를 D마일이라고 두세요
2. 갈 때 걸린 시간을 계산하세요
3. 왕복 전체 평균 속도가 6mph가 되기 위한 총 시간을 계산하세요
4. 이미 사용한 시간과 필요한 총 시간을 비교하세요
5. 남은 시간으로 돌아오는 것이 가능한지 판단하세요
6. 가능하다면 필요한 속도를 계산하고, 불가능하면 왜 불가능한지 논리적으로 설명하세요
"""

In [54]:
print(llm.invoke(prompt).content)

이 문제를 단계적으로 해결해 보겠습니다.

1. **편도 거리를 D마일로 두기**
   - 편도 거리 D = D 마일

2. **갈 때 걸린 시간 계산**
   - 평균 속도 = 3 mph
   - 걸린 시간 = 거리 ÷ 속도 = D ÷ 3 시간

3. **왕복 전체 평균 속도를 6 mph로 만들기 위한 총 시간 계산**
   - 왕복 거리 = 2D
   - 왕복 전체 평균 속도 = 총 거리 ÷ 총 시간
   - 총 시간 = 총 거리 ÷ 왕복 전체 평균 속도 = 2D ÷ 6 = D / 3 시간

4. **이미 사용한 시간과 필요한 총 시간 비교**
   - 이미 사용한 시간 = D ÷ 3 시간
   - 필요한 총 시간 = D / 3 시간
   - 두 시간은 서로 같습니다: D ÷ 3 = D / 3

5. **남은 시간으로 돌아오는 것이 가능한지 판단**
   - 이미 사용한 시간이 필요한 총 시간과 같기 때문에, 돌아올 때 사용하는 시간이 0이 됩니다. 즉, 돌아오는 시간이 없으므로 실제로 돌아올 수 없습니다.

6. **불가능한 이유 설명**
   - 갈 때와 돌아올 때 걸리는 시간의 합이 필요한 총 시간을 초과하기 때문에, 왕복 평균 속도가 6 mph가 될 수 없습니다. 돌아올 때 필요한 속도는 무한대가 되어야 하므로, 현실적으로 돌아오는 것이 불가능합니다.

결론적으로, 왕복 전체 평균 속도를 시속 6마일로 만들기 위해서는 불가능합니다.
